In [1]:
import pandas as pd
from pathlib import Path
from predictors.serialization import load_model

In [2]:
test = pd.read_parquet("../data/processed/test.parquet")

In [3]:
test['multi'] = test['gender'] * 2 + test['age']
labels = { 0: 'Молодая женщина', 1: 'Взрослая женщина', 2: 'Молодой мужчина', 3: 'Взрослый мужчина' }

In [4]:
models_dir = Path("../models")
models = { }

for file in models_dir.glob("cascade_*"):
    model_name = file.name
    models[model_name] = file / "model.pkl"

In [5]:
models

{'cascade_svc_poly_gbm': PosixPath('../models/cascade_svc_poly_gbm/model.pkl'),
 'cascade_svc_poly_svc_rbf': PosixPath('../models/cascade_svc_poly_svc_rbf/model.pkl'),
 'cascade_svc_poly_lr': PosixPath('../models/cascade_svc_poly_lr/model.pkl'),
 'cascade_nb_gbm': PosixPath('../models/cascade_nb_gbm/model.pkl')}

In [6]:
results = pd.DataFrame(index=test.index)

In [7]:
for model in models:
    path = models[model]
    pipeline = load_model(path)
    y_pred = pipeline.predict(test)
    y_proba = pipeline.predict_proba(test)
    
    results[f'{model}_gender_pred_label'] = y_pred[:, 0]
    results[f'{model}_age_pred_label'] = y_pred[:, 1]
    results[f'{model}_multi_label'] = y_pred[:, 0] * 2 + y_pred[:, 1]
    
    for index, (target, cls) in enumerate([
        ('gender', 0),
        ('gender', 1),
        ('age', 0),
        ('age', 1)
    ]):
        results[f'{model}_{target}_pred_proba_{cls}'] = y_proba[:, index]

In [8]:
experiments_dir = Path('../experiments')
experiments_dir.mkdir(parents=True, exist_ok=True)

In [9]:
results.to_parquet(experiments_dir / "test_cascade_pred.parquet")